# Notebook 5: Layer Ablation — Which Layers Matter?

We systematically unfreeze different amounts of the network and measure
the effect. This answers:

- **Are early layers (edges/textures) truly universal?** If unfreezing them
  doesn't help, they were already good enough.
- **Where does domain-specific learning happen?** The layer where unfreezing
  starts helping is where ImageNet features diverge from flower features.
- **Is there a point of diminishing returns?** More trainable params ≠ better.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
from src.data import get_dataloaders
from src.model import create_model, count_parameters
from src.train import train_model

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
train_loader, val_loader, test_loader = get_dataloaders(data_dir='../data', batch_size=32)

In [ ]:
# ---- Run one experiment per unfreezing strategy ----
configs = [
    ("Head only",           "feature_extract", "layer4"),  # baseline
    ("Unfreeze layer4",     "fine_tune",       "layer4"),
    ("Unfreeze layer3+4",   "fine_tune",       "layer3"),
    ("Unfreeze layer2+3+4", "fine_tune",       "layer2"),
    ("Unfreeze all",        "fine_tune",       "layer1"),
]

results = {}
NUM_EPOCHS = 15

for name, mode, unfreeze_from in configs:
    print(f"\n{'='*60}")
    print(f"Experiment: {name}")
    print(f"{'='*60}")

    model = create_model(num_classes=102, mode=mode, unfreeze_from=unfreeze_from)
    model = model.to(DEVICE)
    trainable, total = count_parameters(model)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    history = train_model(
        model, train_loader, val_loader, DEVICE,
        optimizer=optimizer, scheduler=scheduler,
        num_epochs=NUM_EPOCHS,
        save_path=f'../models/ablation_{name.replace(" ", "_")}.pth'
    )

    results[name] = {
        'history': history,
        'trainable_params': trainable,
        'best_val_acc': max(history['val_acc'])
    }

In [ ]:
# ---- Plot comparison ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for name, data in results.items():
    axes[0].plot(data['history']['val_acc'], label=name, linewidth=2)
    axes[1].plot(data['history']['train_acc'], '--', alpha=0.5, linewidth=1)
    axes[1].plot(data['history']['val_acc'], label=name, linewidth=2)

axes[0].set_title('Validation Accuracy by Unfreezing Strategy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Val Accuracy (%)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Train (dashed) vs Val (solid) — Overfitting Check')
axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Bar chart: best val accuracy vs trainable params
names = list(results.keys())
accs = [results[n]['best_val_acc'] for n in names]
params = [results[n]['trainable_params'] for n in names]

ax2 = axes[2]
x = range(len(names))
bars = ax2.bar(x, accs, color='steelblue')
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
ax2.set_ylabel('Best Val Accuracy (%)')
ax2.set_title('Best Accuracy vs Strategy')

# Annotate with param count
for i, (bar, p) in enumerate(zip(bars, params)):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{p/1e6:.1f}M', ha='center', fontsize=8)

ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Layer Ablation Study', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/layer_ablation.png', dpi=150)
plt.show()

## Questions to answer from the results

1. **Does unfreezing layer4 alone give a big jump over head-only?**
   → If yes, the high-level features needed adaptation to flowers.

2. **Does unfreezing layer1+2 on top of layer3+4 help much?**
   → If not, early features (edges, textures) are truly universal.

3. **Which config has the worst train/val gap?**
   → More trainable params + small data = more overfitting risk.

4. **Is there a sweet spot?**
   → Usually unfreezing layer3+4 is optimal for domain shift this size.